In [ ]:
import requests

# 1. Definir la URL de Open-Meteo exacta para Garagoa (Valle de Tenza)
# Coordenadas de Garagoa: Latitud 5.08, Longitud -73.36
# Pedimos: temperatura actual, lluvia (precipitación) y humedad relativa
url = "https://api.open-meteo.com/v1/forecast?latitude=5.08&longitude=-73.36&current=temperature_2m,precipitation,relative_humidity_2m&timezone=America%2FBogota"

# 2. Hacer la petición a la web (Consumo de API)
print("Conectando con Open-Meteo para extraer datos de Garagoa...")
respuesta = requests.get(url, timeout=15)
respuesta.raise_for_status()
datos_crudos = respuesta.json()

# 3. ESTRUCTURA DE DATOS: TABLA HASH (Diccionario en Python para acceso O(1))
tabla_hash_clima = {
    "Garagoa_Nodo_1": {
        "temperatura_C": datos_crudos['current']['temperature_2m'],
        "lluvia_mm": datos_crudos['current']['precipitation'],
        "humedad_relativa_%": datos_crudos['current']['relative_humidity_2m'],
        "ultima_actualizacion": datos_crudos['current']['time']
    }
}

# 4. Imprimir la Tabla Hash (Evidencia para el Dashboard)
print("\n=== DASHBOARD CONTEXTUAL: PREVENCIÓN DE DESLIZAMIENTOS ===")
print("Búsqueda rápida en Tabla Hash O(1) arrojó los siguientes resultados:")
print(tabla_hash_clima["Garagoa_Nodo_1"])

# 5. Lógica de validación cruzada
lluvia_actual = tabla_hash_clima["Garagoa_Nodo_1"]["lluvia_mm"]
humedad_actual = tabla_hash_clima["Garagoa_Nodo_1"]["humedad_relativa_%"]

if lluvia_actual > 0 or humedad_actual > 85:
    print("\n⚠️ ALERTA MACRO-CLIMÁTICA: Lluvia o humedad altísima detectada en la zona.")
    print("El suelo podría estar saturándose de agua. Riesgo de reducción de fricción en la ladera.")
else:
    print("\n✅ Clima externo seco. La saturación de agua por lluvia es nula en este instante.")

In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime

# URL de Open-Meteo para Garagoa pidiendo clima actual + lluvia por hora
url = "https://api.open-meteo.com/v1/forecast?latitude=5.08&longitude=-73.36&current=temperature_2m,precipitation,relative_humidity_2m&hourly=precipitation&past_days=1&forecast_days=1&timezone=America%2FBogota"

num_consultas = 68
historial_registros = []
errores = 0

print(f"Iniciando {num_consultas} consultas reales a Open-Meteo para Garagoa...")

for intento in range(num_consultas):
    try:
        respuesta = requests.get(url, timeout=15)
        respuesta.raise_for_status()
        datos = respuesta.json()

        # Calculamos la lluvia acumulada de las últimas 24 horas sumando la lista horaria
        lluvias_horarias = datos['hourly']['precipitation'][:24]
        lluvia_acumulada_24h = round(sum(lluvias_horarias), 2)

        # ESTRUCTURA DE DATOS: Tabla Hash (Diccionario) para acceso O(1)
        tabla_hash_clima = {
            "Garagoa_Nodo_1": {
                "consulta_num": intento + 1,
                "fecha_hora": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "hora_api": datos['current']['time'],
                "temperatura_C": datos['current']['temperature_2m'],
                "humedad_relativa_%": datos['current']['relative_humidity_2m'],
                "lluvia_instantanea_mm": datos['current']['precipitation'],
                "lluvia_acumulada_24h_mm": lluvia_acumulada_24h
            }
        }

        registro_actual = tabla_hash_clima["Garagoa_Nodo_1"]
        historial_registros.append(registro_actual)

        if (intento + 1) % 10 == 0 or intento == 0 or (intento + 1) == num_consultas:
            print(
                f"Extracción #{intento + 1}/{num_consultas} OK | "
                f"T={registro_actual['temperatura_C']}°C | "
                f"HR={registro_actual['humedad_relativa_%']}% | "
                f"Lluvia24h={registro_actual['lluvia_acumulada_24h_mm']} mm"
            )

    except Exception as e:
        errores += 1
        print(f"Aviso en extracción #{intento + 1}: {e}")

    # Pausa corta para respetar límites de tasa de la API
    if intento + 1 < num_consultas:
        time.sleep(0.5)

# Guardamos el historial acumulado a CSV
df_clima = pd.DataFrame(historial_registros)
df_clima.to_csv("historico_clima_garagoa.csv", index=False)

print("\n=== DATASET CLIMÁTICO ACUMULADO GUARDADO ===")
print(f"Total registros guardados: {len(df_clima)} | Errores: {errores}")
print(df_clima.head(5))
print(df_clima.tail(5))